# Crop Disease Detection — Google Colab Training
EfficientNetB2 with 224x224 input, class imbalance handling, stratified split, advanced augmentation, label smoothing, and per-class metrics.

**Runtime → Change runtime type → T4 GPU**

## Setup:
1. Upload your `Dataset.zip` to Google Drive under `crop-disease-data/`
2. Mount Google Drive
3. Run all cells top to bottom
4. If Colab disconnects mid-training: **just re-run all cells** — it resumes from the last Drive checkpoint
5. Download output from `model_output/`

## 1. Mount Google Drive & GPU Check

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
assert len(tf.config.list_physical_devices('GPU')) > 0, 'No GPU found! Runtime → Change runtime type → T4 GPU'

## 2. Extract Dataset from Google Drive
Upload `Dataset.zip` to `My Drive/crop-disease-data/` first.

In [ ]:
import os, zipfile, shutil, json, math, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.optimizers.schedules import CosineDecay
from sklearn.model_selection import train_test_split
from PIL import Image

DRIVE_ZIP = '/content/gdrive/My Drive/crop-disease-data/Dataset.zip'
EXTRACT_DIR = '/content/crop-dataset'

if not os.path.exists(EXTRACT_DIR):
    assert os.path.exists(DRIVE_ZIP), f'{DRIVE_ZIP} not found! Upload Dataset.zip to Google Drive/crop-disease-data/'
    print(f'Extracting {DRIVE_ZIP}...')
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('Extraction complete')
else:
    print(f'{EXTRACT_DIR} already exists')

# Find the directory that contains class folders
subdirs = sorted([
    d for d in os.listdir(EXTRACT_DIR)
    if os.path.isdir(os.path.join(EXTRACT_DIR, d)) and not d.startswith(('__', '.'))
])

if len(subdirs) == 1:
    # Single subfolder — likely a wrapper; check if it contains class folders
    inner = os.path.join(EXTRACT_DIR, subdirs[0])
    inner_subdirs = [
        d for d in os.listdir(inner)
        if os.path.isdir(os.path.join(inner, d)) and not d.startswith(('__', '.'))
    ]
    if inner_subdirs:
        print(f'Flattening wrapper folder "{subdirs[0]}" with {len(inner_subdirs)} class folders')
        for item in os.listdir(inner):
            shutil.move(os.path.join(inner, item), os.path.join(EXTRACT_DIR, item))
        os.rmdir(inner)

DATASET_DIR = EXTRACT_DIR
classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f'Found {len(classes)} classes:')
for c in classes:
    n = len([f for f in os.listdir(os.path.join(DATASET_DIR, c)) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    print(f'  {c:45s} {n:4d} images')

## 3. Download Real Non-Plant Images for Unknown Class

In [ ]:
import requests

# NOTE: the old source.unsplash.com/random endpoint was retired and now errors,
# so we use Lorem Picsum (picsum.photos) for real non-plant photos instead.
OUT_DIR = os.path.join(DATASET_DIR, 'Unknown___Unknown')
os.makedirs(OUT_DIR, exist_ok=True)

existing = [f for f in os.listdir(OUT_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]
if len(existing) < 200:
    downloaded = 0
    errors = 0
    for i in range(500):
        if downloaded >= 500:
            break
        try:
            url = f'https://picsum.photos/seed/unknown{i}/300/300'
            r = requests.get(url, timeout=10, allow_redirects=True)
            if r.status_code == 200 and len(r.content) > 1000:
                fname = f'picsum_{downloaded:04d}.jpg'
                with open(os.path.join(OUT_DIR, fname), 'wb') as f:
                    f.write(r.content)
                downloaded += 1
                if downloaded % 50 == 0:
                    print(f'Downloaded {downloaded}/500...')
        except Exception:
            errors += 1
            if errors > 150:
                break
    print(f'Downloaded {downloaded} real non-plant images')
    if len([f for f in os.listdir(OUT_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]) < 50:
        print('WARNING: Too few images (no internet?). Generating synthetic fallback.')
        rng = random.Random(42)
        while len([f for f in os.listdir(OUT_DIR) if f.lower().endswith(('.jpg','.jpeg','.png'))]) < 300:
            arr = np.random.randint(0, 256, (224, 224, 3), dtype=np.uint8)
            Image.fromarray(arr).save(os.path.join(OUT_DIR, f'synth_{rng.randint(0,99999):05d}.jpg'))
else:
    print(f'Unknown images already exist ({len(existing)} files)')

## 4. Generate Synthetic Unknown Images (Noise + Leaf-Like Textures)

In [ ]:
from PIL import ImageDraw

OUT_DIR = os.path.join(DATASET_DIR, 'Unknown___Unknown')
rng = random.Random(42)

noise_count = len([f for f in os.listdir(OUT_DIR) if f.startswith('gn_')])
texture_count = len([f for f in os.listdir(OUT_DIR) if f.startswith('lt_')])

if noise_count < 150:
    for sigma in [8, 20, 40, 80, 160]:
        for i in range(40):
            arr = np.clip(np.random.randn(224, 224, 3) * sigma + 128, 0, 255).astype(np.uint8)
            Image.fromarray(arr).save(os.path.join(OUT_DIR, f'gn_{sigma}_{i:04d}.jpg'))
    print(f'Generated 200 Gaussian noise images')

if texture_count < 150:
    for i in range(300):
        bg = rng.randint(30, 90)
        img = Image.new('RGB', (224, 224), (bg, bg + 20, bg))
        draw = ImageDraw.Draw(img)
        for _ in range(rng.randint(3, 12)):
            cx, cy = rng.randint(30, 194), rng.randint(30, 194)
            rx, ry = rng.randint(15, 100), rng.randint(10, 60)
            g = rng.randint(60, 200)
            r = rng.randint(20, min(g, 110))
            b = rng.randint(15, 55)
            draw.ellipse([cx - rx, cy - ry, cx + rx, cy + ry], fill=(r, g, b))
        for _ in range(rng.randint(0, 4)):
            x1, y1, x2, y2 = [rng.randint(0, 224) for _ in range(4)]
            draw.line([(x1, y1), (x2, y2)], fill=(rng.randint(40, 80), rng.randint(60, 120), rng.randint(20, 50)), width=rng.randint(1, 3))
        img.save(os.path.join(OUT_DIR, f'lt_{i:04d}.jpg'))
    print(f'Generated 300 leaf-like texture images')

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
print(f'\nTotal classes: {len(classes)}')

## 5. Training Configuration

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
EPOCHS_PHASE1 = 20
EPOCHS_PHASE2 = 30
LR_PHASE1 = 5e-4
LR_PHASE2 = 1e-4
DROPOUT_RATE = 0.5
LABEL_SMOOTHING = 0.1
WEIGHT_DECAY = 1e-4

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
print('Config set. Classes:', len(classes))

## 6. Stratified Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
base_dir = '/content/data'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

for cls in classes:
    src = os.path.join(DATASET_DIR, cls)
    if not os.path.isdir(src):
        continue
    images = [f for f in os.listdir(src) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    random.shuffle(images)

    n_total = len(images)
    n_train = int(n_total * 0.70)
    n_val = int(n_total * 0.15)
    train_imgs = images[:n_train]
    val_imgs = images[n_train:n_train + n_val]
    test_imgs = images[n_train + n_val:]

    for split_dir, split_imgs in [
        (os.path.join(train_dir, cls), train_imgs),
        (os.path.join(val_dir, cls), val_imgs),
        (os.path.join(test_dir, cls), test_imgs),
    ]:
        os.makedirs(split_dir, exist_ok=True)
        for img in split_imgs:
            shutil.copy2(os.path.join(src, img), os.path.join(split_dir, img))

    print(f'{cls:45s} {len(train_imgs):4d} train, {len(val_imgs):4d} val, {len(test_imgs):4d} test')
print(f'\nTotal classes: {len(classes)}')

## 7. Oversample Minority Classes

In [ ]:
from PIL import ImageEnhance

train_counts = {}
for cls in classes:
    cls_dir = os.path.join(train_dir, cls)
    if os.path.isdir(cls_dir):
        n = len([f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        train_counts[cls] = n

counts = sorted(train_counts.values())
median_count = counts[len(counts)//2]
target_count = int(median_count * 1.2)

print(f'Image counts min: {counts[0]}, median: {median_count}, max: {counts[-1]}')
print(f'Target count per class: {target_count}')

rng = np.random.RandomState(42)

for cls in classes:
    cls_dir = os.path.join(train_dir, cls)
    if not os.path.isdir(cls_dir):
        continue
    n = train_counts.get(cls, 0)
    if n == 0 or n >= target_count:
        continue

    imgs = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    need = target_count - n

    for i in range(need):
        src_img = Image.open(os.path.join(cls_dir, imgs[i % len(imgs)]))
        src_img = src_img.convert('RGB')
        src_img = src_img.resize(IMG_SIZE, Image.LANCZOS)

        if rng.random() < 0.5:
            src_img = src_img.transpose(Image.FLIP_LEFT_RIGHT)
        if rng.random() < 0.4:
            src_img = ImageEnhance.Brightness(src_img).enhance(rng.uniform(0.7, 1.3))
        if rng.random() < 0.4:
            src_img = ImageEnhance.Contrast(src_img).enhance(rng.uniform(0.7, 1.3))
        if rng.random() < 0.3:
            src_img = src_img.rotate(rng.uniform(-25, 25), resample=Image.BICUBIC, fillcolor=(0,0,0))

        ext = os.path.splitext(imgs[i % len(imgs)])[1]
        src_img.save(os.path.join(cls_dir, f'aug_{i:04d}{ext}'))

    print(f'  {cls:45s} {n:4d} -> {target_count:4d}  (+{need} augmented)')

print('Oversampling complete.')

## 8. Compute Class Weights

In [ ]:
class_counts = {}
for i, cls in enumerate(classes):
    cls_dir = os.path.join(train_dir, cls)
    n = len([f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    if n > 0:
        class_counts[i] = n

max_count = max(class_counts.values())
class_weight = {i: max_count / n for i, n in class_counts.items()}

print('Class weights (top 10 heaviest):')
sorted_weights = sorted(class_weight.items(), key=lambda x: -x[1])[:10]
for i, w in sorted_weights:
    print(f'  {classes[i]:45s} weight={w:.2f}')

## 9. Data Augmentation

In [ ]:
def get_train_datagen():
    return tf.keras.preprocessing.image.ImageDataGenerator(
        rotation_range=40,
        width_shift_range=0.25,
        height_shift_range=0.25,
        shear_range=0.2,
        zoom_range=0.3,
        brightness_range=(0.6, 1.4),
        horizontal_flip=True,
        vertical_flip=False,
        fill_mode='reflect',
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    )

def get_eval_datagen():
    return tf.keras.preprocessing.image.ImageDataGenerator(
        preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    )

## 10. Create Data Generators

In [ ]:
train_gen = get_train_datagen().flow_from_directory(
    train_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)
val_gen = get_eval_datagen().flow_from_directory(
    val_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)
test_gen = get_eval_datagen().flow_from_directory(
    test_dir, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

print(f'Training samples: {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Test samples: {test_gen.samples}')

## 11. Build Model (EfficientNetB2)

In [ ]:
base_model = EfficientNetB2(
    include_top=False, weights='imagenet',
    input_shape=(*IMG_SIZE, 3)
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(DROPOUT_RATE)(x)
x = layers.Dense(512, activation='relu', name='dense_hidden',
                 kernel_regularizer=regularizers.l2(WEIGHT_DECAY))(x)
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(
    len(classes), activation='softmax', name='dense_output',
    kernel_regularizer=regularizers.l2(WEIGHT_DECAY)
)(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

## 12. Phase 1 — Train Top Layers
Checkpoints save to Google Drive so they persist across Colab disconnects.

In [ ]:
# Paths on Google Drive (persist across Colab disconnects)
DRIVE_CKPT = '/content/gdrive/My Drive/crop-disease-data/checkpoints'
os.makedirs(DRIVE_CKPT, exist_ok=True)

PHASE1_WEIGHTS = os.path.join(DRIVE_CKPT, 'best_model_phase1.weights.h5')
PHASE2_WEIGHTS = os.path.join(DRIVE_CKPT, 'best_model_phase2.weights.h5')
PHASE2_EPOCHS_FILE = os.path.join(DRIVE_CKPT, 'phase2_epochs_done.txt')

history1 = None
history2 = None

if os.path.exists(PHASE2_WEIGHTS):
    print(f'Found Phase 2 weights on Drive: {PHASE2_WEIGHTS}')
    model.load_weights(PHASE2_WEIGHTS)
    print('Phase 1 SKIPPED  loading Phase 2 weights directly')
elif os.path.exists(PHASE1_WEIGHTS):
    print(f'Found Phase 1 weights on Drive: {PHASE1_WEIGHTS}')
    model.load_weights(PHASE1_WEIGHTS)
    print('Phase 1 SKIPPED  proceeding to Phase 2')
else:
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR_PHASE1),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy']
    )

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=7, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7
        ),
        tf.keras.callbacks.ModelCheckpoint(
            PHASE1_WEIGHTS,
            monitor='val_accuracy', save_best_only=True,
            save_weights_only=True
        ),
    ]

    print('Phase 1: Training top layers...')
    history1 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=EPOCHS_PHASE1, callbacks=callbacks,
        class_weight=class_weight, verbose=1
    )

## 13. Phase 2 — Fine-Tune Entire Model (Resume-Safe)
Saves to Google Drive. If Colab disconnects, just re-run all cells — it resumes from the latest checkpoint.
Tracks completed epochs so remaining epochs are correct on resume.

In [ ]:
# ---- Resume logic ----
epochs_done = 0
if os.path.exists(PHASE2_WEIGHTS):
    print(f'Found existing Phase 2 checkpoint on Drive')
    model.load_weights(PHASE2_WEIGHTS)
    if os.path.exists(PHASE2_EPOCHS_FILE):
        with open(PHASE2_EPOCHS_FILE) as f:
            epochs_done = int(f.read().strip())
    print(f'Resuming: {epochs_done}/{EPOCHS_PHASE2} epochs completed')

remaining = EPOCHS_PHASE2 - epochs_done

if remaining > 0:
    base_model.trainable = True
    for layer in base_model.layers[:100]:
        layer.trainable = False

    total_steps = len(train_gen) * remaining
    cosine_schedule = CosineDecay(
        initial_learning_rate=LR_PHASE2,
        decay_steps=total_steps,
        alpha=0.01
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(cosine_schedule),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
        metrics=['accuracy']
    )

    class EpochTracker(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            total = epochs_done + epoch + 1
            with open(PHASE2_EPOCHS_FILE, 'w') as f:
                f.write(str(total))
            if total % 5 == 0:
                print(f'Checkpoint synced: {total}/{EPOCHS_PHASE2} epochs')

    callbacks2 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=7, restore_best_weights=True
        ),
        tf.keras.callbacks.ModelCheckpoint(
            PHASE2_WEIGHTS,
            monitor='val_accuracy', save_best_only=True,
            save_weights_only=True
        ),
        EpochTracker(),
    ]

    print(f'Phase 2: Fine-tuning {remaining} epochs ({epochs_done} already done)...')
    history2 = model.fit(
        train_gen, validation_data=val_gen,
        epochs=remaining, callbacks=callbacks2,
        class_weight=class_weight, verbose=1
    )
    model.load_weights(PHASE2_WEIGHTS)
    print(f'Phase 2 complete! Best weights saved to Drive.')
else:
    print(f'Phase 2 already completed ({epochs_done}/{EPOCHS_PHASE2} epochs)')

## 14. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report

def clean_name(label):
    name = label.replace('___', ' ').replace('__', ' ').replace('_', ' ')
    return ' '.join(name.split())

test_loss, test_acc = model.evaluate(test_gen, verbose=0)
print(f'Test accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')
print(f'Test loss: {test_loss:.4f}')

all_preds = model.predict(test_gen, verbose=0)
pred_classes = np.argmax(all_preds, axis=1)
true_classes = test_gen.classes
target_names = [clean_name(c) for c in classes]

print()
print(classification_report(true_classes, pred_classes, target_names=target_names, digits=3))

## 15. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

OUTPUT_DIR = '/content/model_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

cm = confusion_matrix(true_classes, pred_classes)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()
print('Confusion matrix saved')

## 16. Save Final Model Weights & Class Names
Copies the best checkpoint from Drive to local output for download.

In [ ]:
OUTPUT_DIR = '/content/model_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

if os.path.exists(PHASE2_WEIGHTS):
    shutil.copy(PHASE2_WEIGHTS, os.path.join(OUTPUT_DIR, 'best_model_phase2.weights.h5'))
    print('Copied best_model_phase2.weights.h5 from Drive to model_output/')
elif os.path.exists(PHASE1_WEIGHTS):
    print('WARNING: Only Phase 1 weights found. Phase 2 did not complete.')

with open(os.path.join(OUTPUT_DIR, 'class_names.json'), 'w') as f:
    json.dump(classes, f, indent=2)
print('Class names saved to model_output/class_names.json')

## 17. Training History Curves

In [ ]:
OUTPUT_DIR = '/content/model_output'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if history1 is not None:
    all_acc = history1.history['accuracy'] + history2.history['accuracy']
    all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
    all_loss = history1.history['loss'] + history2.history['loss']
    all_val_loss = history1.history['val_loss'] + history2.history['val_loss']
    phase1_end = len(history1.history['accuracy'])
else:
    all_acc = history2.history['accuracy']
    all_val_acc = history2.history['val_accuracy']
    all_loss = history2.history['loss']
    all_val_loss = history2.history['val_loss']
    phase1_end = 0

axes[0].plot(all_acc, label='Train Accuracy')
axes[0].plot(all_val_acc, label='Val Accuracy')
if phase1_end > 0:
    axes[0].axvline(x=phase1_end, color='r', linestyle='--', alpha=0.5, label='Phase 2 start')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(all_loss, label='Train Loss')
axes[1].plot(all_val_loss, label='Val Loss')
if phase1_end > 0:
    axes[1].axvline(x=phase1_end, color='r', linestyle='--', alpha=0.5, label='Phase 2 start')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_history.png'), dpi=150)
plt.show()
print('Training history saved')

## 18. Download Output Files
Copy these to your project:
- `best_model_phase2.weights.h5` → `backend/best_model/best_model_phase2.weights.h5`
- `class_names.json` → `backend/class_names.json`

In [ ]:
from google.colab import files

OUTPUT_DIR = '/content/model_output'

print('Output files:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f:40s} {size / 1024 / 1024:.2f} MB')

print()
print('Downloading best_model_phase2.weights.h5...')
files.download(os.path.join(OUTPUT_DIR, 'best_model_phase2.weights.h5'))
print('Downloading class_names.json...')
files.download(os.path.join(OUTPUT_DIR, 'class_names.json'))
print('Done!')